In [ ]:
# ==============================================================================
# CELL 1 / PHASE 1: SNOMED CT INGESTION
# 
# ARCHITECTURE NOTE: SNOMED CT is a massive, logically defined clinical ontology.
# It is built on Description Logic, meaning concepts are organized by taxonomic 
# parents and associative attributes (e.g., "Roman Catholic" IS_A "Christian" 
# AND Interprets "Social / personal history observable").
#
# SSSOM ALIGNMENT STRATEGY: 
# To navigate this structure efficiently, the script uses a two-phase API 
# strategy (Strategy B). 
# 1. Bulk Discovery: It uses SNOMED's Expression Constraint Language (ECL) via 
#    the FHIR API (`ecl/<<{seed_id}`) to request all descendants of target seeds.
# 2. Detail Extraction: It iterates through the discovered IDs, querying the 
#    Snowstorm Browser API to extract the full JSON metadata.
# 
# To preserve the native Description Logic for downstream mapping, the script 
# extracts the `definitionStatus` and maps it to our `Concept_Type` column 
# (e.g., `owl:Class (Primitive)` vs `owl:Class (Fully Defined)`).
# ==============================================================================

import requests
import pandas as pd
import time
import datetime
import os
import sys
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
load_dotenv("../../config/.env") 
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

raw_ingest_file = os.path.join(raw_data_dir, "raw_snomed.csv")

# Import Schema Manager
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory. Check PYTHONPATH.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "SNOMED"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="SNOMED CT",
    fallback_uri="http://snomed.info/id/"
)
SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']

HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 (Research; mailto:{contact_email})',
    'Accept': 'application/json'
}

# --- 3. Request & Hierarchy Functions ---
def safe_request(url, params=None):
    """Executes an API request with retries and politeness delays."""
    for attempt in range(3):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=15)
            if response.status_code == 200: return response.json()
            elif response.status_code == 404: return None
            time.sleep(2)
        except Exception:
            time.sleep(2)
    return None

path_cache = {}
def get_full_path(sctid, depth=0):
    """Recursively queries the 'Is a' relationships to build an absolute path to the SNOMED root."""
    sctid = str(sctid)
    if sctid in path_cache: return path_cache[sctid]
    if depth > 10: return "!!! DEPTH LIMIT REACHED !!!"
    if sctid == '138875005': return "" # SNOMED Root Concept ID
    
    browser_url = f"https://snowstorm.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/concepts/{sctid}"
    res = safe_request(browser_url)
    if not res: return sctid
        
    term = res.get('pt', {}).get('term', sctid)
    # Extract only the active 'Is a' (116680003) relationships for the taxonomic path
    parents = [r['destinationId'] for r in res.get('relationships', []) if r.get('active') and r.get('typeId') == '116680003']
    
    if not parents:
        path_cache[sctid] = term
        return term
    
    parent_path = get_full_path(parents[0], depth + 1)
    if "DEPTH LIMIT REACHED" in parent_path: return "!!! DEPTH LIMIT REACHED !!!"

    full_p = f"{parent_path} > {term}" if parent_path else term
    path_cache[sctid] = full_p
    return full_p

# --- 4. Persistent Progress Tracking & Schema Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        all_processed_ids = set()
    else:
        all_processed_ids = set(existing_df['Concept_ID'].tolist())
        print(f"[*] Resuming: Found {len(all_processed_ids)} SNOMED concepts already saved.")
else:
    all_processed_ids = set()
    print("[*] Starting fresh SNOMED CT ingest.")

# --- 5. Main Execution ---
SEED_CONCEPTS = [
    "105390002",  # Spiritual therapy / Spiritual care
    "11015003",   # Minister of religion / clergy
    "1400009",    # Religion and philosophy (Root)
    "183945002",  # Procedure declined for religious reason
    "410597007",  # Person by affiliation with belief system
    "408892004"   # Finding of religious/philosophical belief
]

for seed_id in SEED_CONCEPTS:
    print(f"\n--- Starting Harvest for Seed: {seed_id} ---")
    
    # Use FHIR ECL to find all descendants of the seed
    fhir_url = "https://snowstorm.ihtsdotools.org/fhir/ValueSet/$expand"
    data = safe_request(fhir_url, params={'url': f'http://snomed.info/sct?fhir_vs=ecl/<<{seed_id}', 'count': 1000})

    if not data or 'expansion' not in data:
        print(f"Expansion failed or empty for {seed_id}. Skipping.")
        continue

    items = data.get('expansion', {}).get('contains', [])
    initial_list = [{'Concept_ID': str(item.get('code')), 'Primary_Label': item.get('display')} for item in items if item.get('code')]
    total = len(initial_list)
    
    print(f"Found {total} concepts. Enriching and appending to CSV...")
    browser_base = "https://snowstorm.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/concepts"
    
    for i, entry in enumerate(initial_list, 1):
        cid = entry['Concept_ID']
        if cid in all_processed_ids: continue
            
        print(f"\r[{i}/{total}] Processing: {entry['Primary_Label'][:50]} ({cid}){ ' ' * 20 }", end="", flush=True)
        
        # Pull the full JSON payload for the specific concept
        res = safe_request(f"{browser_base}/{cid}")
        
        if res:
            fsn = res.get('fsn', {}).get('term', '')
            def_status = res.get('definitionStatus', '')
            
            # Map Description Logic status to standard W3C Representational Semantics
            if "(attribute)" in fsn.lower() or "(linkage concept)" in fsn.lower(): concept_type = "owl:Property"
            elif def_status == "FULLY_DEFINED": concept_type = "owl:Class (Fully Defined)"
            else: concept_type = "owl:Class (Primitive)"
                
            syns, defs = [], []
            has_translation = ""
            
            for d in res.get('descriptions', []):
                if not d.get('active'): continue
                lang = d.get('lang', '').lower()
                if not lang or lang.startswith('en'):
                    if d.get('type') == 'SYNONYM': syns.append(d.get('term', ''))
                    elif d.get('type') == 'TEXT_DEFINITION': defs.append(d.get('term', ''))
                else: has_translation = "yes"

            # Extract ONLY taxonomic parents for the Parent_IDs column
            parents = [r['destinationId'] for r in res.get('relationships', []) if r.get('active') and r.get('typeId') == '116680003']
            
            extracted_data = {
                'Source_System': SOURCE_NAME,
                'Primary_Label': res.get('pt', {}).get('term', entry['Primary_Label']),
                'CURIE': f"{SOURCE_PREFIX}:{cid}",
                'Formal_Label': fsn,
                'Concept_Type': concept_type,
                'Hierarchy_Path': get_full_path(cid),
                'Synonyms': " | ".join(list(set(syns))),
                'Description': " ".join(list(set(defs))), 
                'Parent_IDs': " | ".join([str(p) for p in parents]),
                'Concept_ID': str(cid),
                'URI': f"{BASE_URI}{cid}",
                'Has_Translation': has_translation,
                'Status': "active" if res.get('active') else "inactive",
                'Crosswalks': "" # SNOMED external mappings live in separate reference sets (RefSets)
            }
            
            clean_row = finalize_row(extracted_data)
            pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
                raw_ingest_file, mode='a', index=False, 
                header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
            )
            all_processed_ids.add(cid)
        else:
            print(f"\n[!] Skipping {cid} (API returned no data).")

        time.sleep(1.2) # API Politeness

print(f"\n\nDone! Total unique concepts processed: {len(all_processed_ids)}")